In [1]:
# Chunk 1: Load raw data
import pandas as pd
import numpy as np

# Load the raw dataset
raw_df = pd.read_csv("../Data/Raw/telecom_call_drop.csv")

# Quick look
print("Raw data shape:", raw_df.shape)
print(raw_df.head())
print(raw_df['call_id'].nunique(), "unique calls")

Raw data shape: (259333, 10)
    timestamp  rsrp  rsrq  sinr  rain_intensity  call_id    band  tower_load  \
0  1700000000 -70.5 -10.7  18.7             0.0        0  LTE_B3          62   
1  1700000001 -69.3  -7.7  11.1             0.0        0  LTE_B3          62   
2  1700000002 -69.5  -9.4  21.3             0.0        0  LTE_B3          62   
3  1700000003 -69.8  -5.2  22.6             0.0        0  LTE_B3          62   
4  1700000004 -68.5  -3.0  19.2             0.0        0  LTE_B3          62   

   speed_kmph  is_drop  
0         2.3        0  
1         2.3        0  
2         2.3        0  
3         2.3        0  
4         2.3        0  
2000 unique calls


In [2]:
# Chunk 2: Ensure correct order within each call
raw_df = raw_df.sort_values(['call_id', 'timestamp']).reset_index(drop=True)
print("Data sorted.")

Data sorted.


In [3]:
# Chunk 3: Helper functions used in aggregation
def last_n_slope(series, n=5):
    """
    Slope of the last `n` values (linear regression).
    Positive slope = signal improving, negative = degrading.
    """
    if len(series) < 2:
        return 0.0
    y = series.iloc[-n:].values if len(series) >= n else series.values
    x = np.arange(len(y))
    slope = np.polyfit(x, y, 1)[0]
    return slope

def time_below_threshold(series, threshold):
    """Number of seconds where series < threshold."""
    return (series < threshold).sum()

def compute_duration(ts_series):
    """Call duration in seconds (max timestamp - min timestamp)."""
    return ts_series.max() - ts_series.min()

In [4]:
# Chunk 4: Replace impossible signal values with realistic floors
# -999 often means "no signal" (typical at the end of a dropped call)
# Replace them so that calls are not removed later.

# Replace -999 in RSRP with -130 (very weak but still plausible)
raw_df['rsrp'] = raw_df['rsrp'].replace(-999, -130)

# For RSRQ, typical range -3 to -20; -999 is invalid, set to -20 (worst)
raw_df['rsrq'] = raw_df['rsrq'].replace(-999, -20)

# For SINR, typical -10 to 30; -999 -> -5 (very poor)
raw_df['sinr'] = raw_df['sinr'].replace(-999, -5)

# Rain intensity: -999 is impossible; replace with 0
raw_df['rain_intensity'] = raw_df['rain_intensity'].replace(-999, 0)

print("Invalid signal values replaced with realistic floors.")

Invalid signal values replaced with realistic floors.


In [5]:
# Chunk 5 (modified): Aggregation dictionary including tower_load and speed_kmph
grouped = raw_df.groupby('call_id')

agg_funcs = {
    # RSRP features
    'rsrp_min': ('rsrp', 'min'),
    'rsrp_max': ('rsrp', 'max'),
    'rsrp_mean': ('rsrp', 'mean'),
    'rsrp_std': ('rsrp', 'std'),
    'rsrp_last': ('rsrp', 'last'),
    'rsrp_slope_last5': ('rsrp', lambda x: last_n_slope(x, 5)),
    'rsrp_slope_all': ('rsrp', lambda x: last_n_slope(x, len(x))),
    'rsrp_time_below_minus110': ('rsrp', lambda x: time_below_threshold(x, -110)),

    # RSRQ features
    'rsrq_min': ('rsrq', 'min'),
    'rsrq_mean': ('rsrq', 'mean'),
    'rsrq_std': ('rsrq', 'std'),
    'rsrq_slope_last5': ('rsrq', lambda x: last_n_slope(x, 5)),

    # SINR features
    'sinr_min': ('sinr', 'min'),
    'sinr_mean': ('sinr', 'mean'),
    'sinr_time_below_0': ('sinr', lambda x: time_below_threshold(x, 0)),

    # Rain intensity
    'rain_mean': ('rain_intensity', 'mean'),
    'rain_max': ('rain_intensity', 'max'),

    # --- NEW: Tower load (constant per call, so any aggregate works) ---
    'tower_load_mean': ('tower_load', 'mean'),
    'tower_load_max': ('tower_load', 'max'),   # same as mean, but for consistency
    'tower_load_high': ('tower_load', lambda x: (x > 80).sum()),  # seconds above 80% load

    # --- NEW: Speed (km/h) - constant per call ---
    'speed_kmph_mean': ('speed_kmph', 'mean'),
    'speed_kmph_max': ('speed_kmph', 'max'),

    # Temporal features
    'call_duration_sec': ('timestamp', compute_duration),
    'start_timestamp': ('timestamp', 'min'),
}

features = grouped.agg(**agg_funcs).reset_index()   # call_id becomes a column

In [6]:
# Chunk 6: Extract hour of day from start_timestamp
features['start_hour'] = pd.to_datetime(features['start_timestamp'], unit='s').dt.hour
features.drop('start_timestamp', axis=1, inplace=True)

In [7]:
# Chunk 7: Convert band to dummy variables (0/1 integers)
band_info = raw_df[['call_id', 'band']].drop_duplicates('call_id')
band_dummies = pd.get_dummies(band_info['band'], prefix='band', dtype=int)
band_dummies['call_id'] = band_info['call_id']

features = features.merge(band_dummies, on='call_id', how='left')

In [8]:
# Chunk 8: Attach target (is_drop) – one value per call
target_df = raw_df[['call_id', 'is_drop']].drop_duplicates('call_id')
features = features.merge(target_df, on='call_id', how='left')

In [9]:
# Chunk 9 (optional): Only remove truly impossible values (e.g., rsrp_min < -140)
# But do NOT remove dropped calls – they have low RSRP by definition.
# We skip filtering entirely because we already replaced -999.
# If you still want to remove extreme outliers (e.g., below -140), do it carefully:
# features = features[features['rsrp_min'] > -140]

print(f"Before any filtering: {features.shape[0]} calls, drops = {features['is_drop'].sum()}")

Before any filtering: 2000 calls, drops = 140


In [10]:
# Chunk 10: Force all columns to numeric, drop non‑convertible, save
# First, drop the 'call_id' column (not a feature)
features = features.drop('call_id', axis=1)

# Convert every column to numeric, coercing errors to NaN
for col in features.columns:
    features[col] = pd.to_numeric(features[col], errors='coerce')

# Drop any column that now contains only NaN (or has zero variance)
nan_cols = features.columns[features.isna().all()].tolist()
if nan_cols:
    print(f"Dropping columns with all NaN: {nan_cols}")
    features = features.drop(columns=nan_cols)

# Ensure the target is integer 0/1 (it already is, but just in case)
features['is_drop'] = features['is_drop'].astype(int)

# Check that all remaining columns are numeric
non_numeric = features.select_dtypes(exclude=['number']).columns.tolist()
if non_numeric:
    print(f"Warning: non‑numeric columns still present: {non_numeric}")
    # Option: drop them or convert again
    features = features.drop(columns=non_numeric)
else:
    print("All columns are numeric.")

# Save engineered dataset
features.to_csv("../Data/Engineered/Engineered.csv", index=False)
print(f"Saved engineered data with {features.shape[0]} rows (calls) and {features.shape[1]} columns.")
print("Sample of first 2 rows:\n", features.head(2))

All columns are numeric.
Saved engineered data with 2000 rows (calls) and 28 columns.
Sample of first 2 rows:
    rsrp_min  rsrp_max  rsrp_mean  rsrp_std  rsrp_last  rsrp_slope_last5  \
0     -86.3     -68.0 -79.745902  4.866586      -84.0             -0.03   
1     -80.5     -64.5 -73.235043  4.846056      -76.9              0.59   

   rsrp_slope_all  rsrp_time_below_minus110  rsrq_min  rsrq_mean  ...  \
0       -0.121795                         0     -14.8  -8.442623  ...   
1       -0.061442                         0     -16.0  -8.335043  ...   

   tower_load_max  tower_load_high  speed_kmph_mean  speed_kmph_max  \
0              62                0              2.3             2.3   
1              51                0             48.5            48.5   

   call_duration_sec  start_hour  band_5G_n78  band_LTE_B20  band_LTE_B3  \
0                121          22            0             0            1   
1                233          22            0             0            1   



In [11]:
# Chunk 11: Final verification and class distribution
print("\nData types after conversion:")
print(features.dtypes.value_counts())

print("\nClass distribution after processing:")
print(features['is_drop'].value_counts())
print("\nPercentages:")
print(features['is_drop'].value_counts(normalize=True) * 100)
print("\nClass distribution after processing:")
print(features['is_drop'].value_counts())
print("\nPercentages:")
print(features['is_drop'].value_counts(normalize=True) * 100)


Data types after conversion:
float64    18
int64       9
int32       1
Name: count, dtype: int64

Class distribution after processing:
is_drop
0    1860
1     140
Name: count, dtype: int64

Percentages:
is_drop
0    93.0
1     7.0
Name: proportion, dtype: float64

Class distribution after processing:
is_drop
0    1860
1     140
Name: count, dtype: int64

Percentages:
is_drop
0    93.0
1     7.0
Name: proportion, dtype: float64
